#1 **Programmation en RDDs (PySpark) - Python**

In [7]:
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark import SparkConf, SparkContext


spark = SparkSession.builder.appName("CCF Correct RDD").getOrCreate()
sc = spark.sparkContext


new_pairs_counter = sc.accumulator(0)

def ccf_correct_implementation(edges_rdd, max_iters=20):

    global new_pairs_counter


    nodes = edges_rdd.flatMap(lambda edge: [edge[0], edge[1]]).distinct()
    node_component_rdd = nodes.map(lambda node: (node, node))  # (nœud, id_de_composante)


    neighbors_rdd = edges_rdd.flatMap(lambda x: [(x[0], x[1]), (x[1], x[0])])

    iteration = 0
    start_time = time.time()


    while iteration < max_iters:
        iteration += 1
        print(f" Démarrage de l'itération {iteration}...")


        new_pairs_counter.value = 0


        joined_rdd = node_component_rdd.join(neighbors_rdd).map(
            lambda x: (x[1][1], x[1][0])
        )

        input_for_reducer = joined_rdd.union(node_component_rdd).groupByKey()


        def ccf_iterate_reducer(key_values):
            key, values_iter = key_values
            values = list(values_iter)
            min_val = min(values)

            if min_val < key:

                yield (key, min_val)
                for val in values:
                    if val != min_val:
                        new_pairs_counter.add(1)
                        yield (val, min_val)
            else:

                yield (key, key)


        ccf_iterate_output = input_for_reducer.flatMap(ccf_iterate_reducer)
        dedup_output = ccf_iterate_output.distinct()


        node_component_rdd = dedup_output


        node_component_rdd.count()


        if new_pairs_counter.value == 0:
            print(f"Convergence atteinte en {iteration} itérations.")
            break

    exec_time = time.time() - start_time
    return node_component_rdd, iteration, exec_time


schema = StructType([
    StructField("source", IntegerType(), True),
    StructField("target", IntegerType(), True)
])


files = [
    ("G1_1k.csv", "G1"),
    ("G2_5k.csv", "G2"),
    ("G3_8k.csv", "G3"),
    ("G4_10k.csv", "G4")
]


results = []

for filename, label in files:
    filepath = f"data/{filename}"
    print(f"📎 Traitement de {label} ({filepath})...")

    try:

        df = spark.read.csv(filepath, header=True, schema=schema)

        edges_rdd = df.rdd.map(lambda row: (row['source'], row['target']))


        components, num_iters, exec_time = ccf_correct_implementation(edges_rdd, max_iters=20)

        nb_nodes = edges_rdd.flatMap(lambda edge: [edge[0], edge[1]]).distinct().count()
        nb_edges = edges_rdd.count()

        print(f" Données du graphe: {nb_nodes} nœuds, {nb_edges} arêtes")
        print(f" Itérations : {num_iters}")
        print(f" Temps : {round(exec_time, 3)} secondes")
        print("-" * 40)


        results.append((label, nb_nodes, nb_edges, num_iters, round(exec_time, 3)))

    except Exception as e:
        print(f" Erreur avec {label} : {e}")
        print("-" * 40)


result_rdd_df = pd.DataFrame(
    results,
    columns=["Graphe", "Nœuds", "Arêtes", "Itérations", "Temps (s)"]
)

print(" Résumé des performances (RDD) :")
print(result_rdd_df)

📎 Traitement de G1 (data/G1_1k.csv)...
 Erreur avec G1 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G1_1k.csv.
----------------------------------------
📎 Traitement de G2 (data/G2_5k.csv)...
 Erreur avec G2 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G2_5k.csv.
----------------------------------------
📎 Traitement de G3 (data/G3_8k.csv)...
 Erreur avec G3 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G3_8k.csv.
----------------------------------------
📎 Traitement de G4 (data/G4_10k.csv)...
 Erreur avec G4 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G4_10k.csv.
----------------------------------------
 Résumé des performances (RDD) :
Empty DataFrame
Columns: [Graphe, Nœuds, Arêtes, Itérations, Temps (s)]
Index: []


# **2 	Implémentation CCF avec DataFrames _ Python**

In [8]:
import time
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, least
from pyspark.sql.types import StructType, StructField, IntegerType


spark = SparkSession.builder \
    .appName("CCF DataFrame Correct") \
    .getOrCreate()

def ccf_dataframe_implementation(edges_df, max_iters=20):



    nodes = edges_df.select("source").union(edges_df.select("target")) \
        .distinct() \
        .withColumnRenamed("source", "node")

    labels = nodes.withColumn("component_id", col("node"))

    iteration = 0
    start_time = time.time()


    adj_list = edges_df.select("source", "target").union(
        edges_df.select(col("target").alias("source"), col("source").alias("target"))
    )

    while iteration < max_iters:
        iteration += 1
        print(f" Démarrage de l'itération {iteration}...")


        labels_renamed = labels.withColumnRenamed("component_id", "current_component_id")


        new_labels = adj_list.join(labels_renamed, adj_list.target == labels_renamed.node) \
            .select(
                adj_list.source.alias("node"),
                labels_renamed.current_component_id.alias("neighbor_component_id")
            ) \
            .groupBy("node") \
            .agg({"neighbor_component_id": "min"}) \
            .withColumnRenamed("min(neighbor_component_id)", "propagated_id")


        current_and_new_labels = labels_renamed.join(new_labels, "node", "left_outer")


        updated_labels = current_and_new_labels \
            .withColumn(
                "new_label",
                least(col("current_component_id"), col("propagated_id"))
            ) \
            .select(col("node"), col("new_label").alias("component_id"))


        changes = updated_labels \
            .join(labels.withColumnRenamed("component_id", "old_component_id"), "node") \
            .filter(col("component_id") != col("old_component_id")) \
            .count()


        labels = updated_labels

        if changes == 0:
            print(f"Convergence atteinte en {iteration} itérations.")
            break

    exec_time = time.time() - start_time
    return labels, iteration, exec_time



schema = StructType([
    StructField("source", IntegerType(), True),
    StructField("target", IntegerType(), True)
])


files = [
    ("G1_1k.csv", "G1"),
    ("G2_5k.csv", "G2"),
    ("G3_8k.csv", "G3"),
    ("G4_10k.csv", "G4")
]


results = []


for filename, label in files:
    filepath = f"data/{filename}"
    print(f"📎 Traitement de {label} ({filepath})...")

    try:

        edges_df = spark.read.csv(filepath, header=True, schema=schema)


        components_df, num_iters, exec_time = ccf_dataframe_implementation(edges_df, max_iters=20)


        nb_nodes = edges_df.select("source").union(edges_df.select("target")).distinct().count()
        nb_edges = edges_df.count()

        print(f" Données du graphe : {nb_nodes} nœuds, {nb_edges} arêtes")
        print(f" Itérations : {num_iters}")
        print(f"⏱ Temps d'exécution : {round(exec_time, 3)} secondes")
        print("-" * 40)

        results.append((label, nb_nodes, nb_edges, num_iters, round(exec_time, 3)))

    except Exception as e:
        print(f" Erreur lors du traitement de {label} : {e}")
        print("-" * 40)

result_df = pd.DataFrame(
    results,
    columns=["Graphe", "Nœuds", "Arêtes", "Itérations", "Temps (s)"]
)

print(" Résumé des performances (DataFrame) :")
print(result_df)


📎 Traitement de G1 (data/G1_1k.csv)...
 Erreur lors du traitement de G1 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G1_1k.csv.
----------------------------------------
📎 Traitement de G2 (data/G2_5k.csv)...
 Erreur lors du traitement de G2 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G2_5k.csv.
----------------------------------------
📎 Traitement de G3 (data/G3_8k.csv)...
 Erreur lors du traitement de G3 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G3_8k.csv.
----------------------------------------
📎 Traitement de G4 (data/G4_10k.csv)...
 Erreur lors du traitement de G4 : [PATH_NOT_FOUND] Path does not exist: file:/content/data/G4_10k.csv.
----------------------------------------
 Résumé des performances (DataFrame) :
Empty DataFrame
Columns: [Graphe, Nœuds, Arêtes, Itérations, Temps (s)]
Index: []
